In [1]:
import pandas as pd
import numpy as np 
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.impute import SimpleImputer

In [2]:
# 사이킷런을 이용해 당뇨병 데이터 다운로드
diabetes = fetch_openml(
    data_id=37,
    as_frame=True,
    parser="pandas"
)

# 특성 데이터 복사
df = diabetes.data.copy()

# 정답 추가 (0 = 음성, 1 = 양성)
df["target"] = diabetes.target.map({
    "tested_negative": 0,
    "tested_positive": 1
}).astype(int)

print(df.shape)
print(df.head(3))


(768, 9)
   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1


## 1.데이터 탐색

In [3]:

print(df.shape)
print(df.head())
df.info()
df.describe()

print(df["target"].value_counts())
print("음성0 양성1 비율:",round(df["target"].value_counts(normalize = True)*100,2))
df.describe()


(768, 9)
   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1
3     1    89    66    23    94  28.1  0.167   21       0
4     0   137    40    35   168  43.1  2.288   33       1
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    768 non-null    int64  
 2   pres    768 non-null    int64  
 3   skin    768 non-null    int64  
 4   insu    768 non-null    int64  
 5   mass    768 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   target  768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
target
0    500
1    268
Name: count, dtype: int64
음성0 양성1 비율: target
0    65.1
1    34.9
Na

,preg,plas,pres,skin,insu,mass,pedi,age,target
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


- `plas` (혈당)
- `pres` (혈압)
- `skin` (피부 두께)
- `insu` (인슐린)
- `mass` (BMI)
- `preg` (임신 횟수) 
- `pedi` (당뇨병 가족력 관련 지표)
- `age`	 (나이)

In [4]:
d = df.drop(columns="target")
# 정답인 타겟 제외
r = d.max() - d.min()
print(r.idxmax(),r.max())
print(r.idxmin(), r.min())
# 격차 최대 최소 찾기

insu 846.0
pedi 2.342


## 분석

1. 양성 34.9% 음성 65.1% 
2. 범위가 가장 큰 특성 - insu 인슐린  , 가장 작은 특성 - pedi 가족지표
3. KNN은 데이터 간의 거리를 계산하는 알고리즘, 특성마다 숫자 범위 차이가 크면 범위가 큰 특성(insu) 가 과도하게 반영되고 범위가 작은 특성(pedi)의 영향이 과소되거나 무시되어 예측 성능이 떨어짐 따라서 스케일링이 필요 (standardscaler, minmaxscaler) 

## 2.결측치 탐색

In [5]:

print(df.isna().sum())
# 결측치 0
print("0의 개수:",(df[["plas","pres","skin","insu","mass"]] == 0).sum())
 
df[["plas","pres","skin","insu","mass"]] = df[["plas","pres","skin","insu","mass"]].replace(0,np.nan)
# 0을 결측치 nan으로 채우기

df.info()

preg      0
plas      0
pres      0
skin      0
insu      0
mass      0
pedi      0
age       0
target    0
dtype: int64
0의 개수: plas      5
pres     35
skin    227
insu    374
mass     11
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    763 non-null    float64
 2   pres    733 non-null    float64
 3   skin    541 non-null    float64
 4   insu    394 non-null    float64
 5   mass    757 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   target  768 non-null    int64  
dtypes: float64(6), int64(3)
memory usage: 54.1 KB


In [6]:
print("결측수:",df.isna().sum())
print("결측 행 수:",df.isna().any(axis=1).sum())
# 결측치 0으로 출력됨


결측수: preg        0
plas        5
pres       35
skin      227
insu      374
mass       11
pedi        0
age         0
target      0
dtype: int64
결측 행 수: 376


## 분석
1. 측정되지 않은 실제 결측치들을 데이터 수집 과정에서 임의로 숫자 0으로 기록, 판다스의 isna()는 NaN이나 None만 결측치로 인식, 숫자 0을 정상적인 데이터로 판단해 탐지하지 못햇음.

2. preg의 정상적인 데이터0과 target컬럼에서 음성을 나타내는 0도 결측치로 대체하기에 결측치로 처리할 부분만 대체


## 3.특성과 정답 분리 및 데이터 분할

In [7]:

x = df.drop(columns="target")
y= df["target"]

print(x.shape)
print(y.shape)

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42, stratify=y)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(768, 8)
(768,)
(614, 8)
(154, 8)
(614,)
(154,)


## 분석 
1. 모델이 학습에 사용되지 않은 일반 데이터에서 잘 작동하는지 일반화 성능을 평가하기 위함, 또한 학습 데이터만을 숙달한 데이터 과적합을 방지.판별하기 위함이기도 함

2. 원본 데이터의 비율을 학습과 테스트 데이터에도 동일하게 비율을 유지하기 위함 - 무작위시 특정 비율로 왜곡될 우려 

3. 난수 시드를 고정하여 항상 동일한 방식으로 데이터가 분할되도록 설계, 이로 인해 k값이나 모델의 변경 등으로 인한 변수가 발생시 순수하게 모델과 k값의 변동으로 인한 성능 비교.검증을 공정하게 할 수 있음

## 4.결측치 처리

In [8]:

x_median = x_train.median()
x_train_filled = x_train.fillna(x_median)
x_test_filled = x_test.fillna(x_median)

# imputer = SimpleImputer(strategy="median")
# x_train = imputer.fit_transform(x_train)
# x_test = imputer.transform(x_test)

print("훈련:",x_train_filled.isna().sum())
print("시험:",x_test_filled.isna().sum())

print("훈련 처리 전:",x_train.shape)
print("훈련 처리 후:",x_train_filled.shape)
print("시험 처리 전:",x_test.shape)
print("시험 처리 후:",x_test_filled.shape)
# 데이터 손실 없이 중앙값으로 채운 데이터를 카피본에 넣음

훈련: preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64
시험: preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64
훈련 처리 전: (614, 8)
훈련 처리 후: (614, 8)
시험 처리 전: (154, 8)
시험 처리 후: (154, 8)


## 분석

1. 중앙값을 사용하여 이상치로 인한 데이터 왜곡을 방지하기 위함이고, 학습데이터 대폭 삭제로 인해 모델 성능의 하락을 방지하기 위함

2. 테스트 데이터의 중앙값을 사용하면 먼저, 테스트 데이터를 모델이 미리 학습하는 데이터 누수가 발생해 모델의 성능 평가 신뢰성이 하락하고, 또한 실제 환경에서 새로운 데이터가 1건씩 입력되므로 중앙값 계산이 불가하다.

## 5. 첫 번째 KNN 학습

In [9]:


model = KNeighborsClassifier(n_neighbors=5)
model.fit(x_train_filled,y_train)
print("학습 정확도 :",model.score(x_train_filled,y_train))

model.predict(x_test_filled)
print("테스트 정확도 :",model.score(x_test_filled,y_test))

print("예측 결과 :",model.predict(x_test_filled)[:10])
print("실제 정답 :",y_test.values[:10])
# idx제거 해서 직관성 높이기


학습 정확도 : 0.8013029315960912
테스트 정확도 : 0.6753246753246753
예측 결과 : [1 0 0 0 0 0 0 1 0 1]
실제 정답 : [0 0 0 1 0 0 1 1 0 0]


## 분석

1. 데이터 예측시 거리가 가까운 5개의 이웃 데이터를 참고한다는 의미 이진 분류에서 동점 발생을 위해 주로 홀수를 사용하며 k값이 너무 작으면 데이터 과적합, 너무 크면 과소적합 위험 5가 기본값이자 가장 이상적인 수치 

2. 학습정확도: 학습데이터를 사용한 정답률 ,모델이 학습 데이터 패턴을 얼마나 익혓는지를 의미하는  지표
   테스트 정확도: 학습에 사용되지 않은 새로운 데이터로 예측시의 정답률, 실제 환경에서의 일반화 성능 

3. 학습 데이터는 모델이 이미 패턴과 정답을 익힌 데이터인 반면, 테스트 데이터는 새로운 데이터 이기 때문이며, 특히 지금처럼 스케일링을 하지 않아 특성 간 스케일 차이가 있는 경우 일반화 성능이 더욱 떨어진다 


## 6. KNN 성능 평가

In [10]:

pred = model.predict(x_test_filled)
print("혼동행렬:", confusion_matrix(y_test, pred))
print("정확도:", accuracy_score(y_test, pred) )
print("정밀도:", precision_score(y_test, pred))
print("재현율:", recall_score(y_test,pred))
print("F1점수:", f1_score(y_test,pred))
print(classification_report(y_test,pred))




혼동행렬: [[77 23]
 [27 27]]
정확도: 0.6753246753246753
정밀도: 0.54
재현율: 0.5
F1점수: 0.5192307692307693
              precision    recall  f1-score   support

           0       0.74      0.77      0.75       100
           1       0.54      0.50      0.52        54

    accuracy                           0.68       154
   macro avg       0.64      0.64      0.64       154
weighted avg       0.67      0.68      0.67       154



## 분석

1. 27개
2. 23개
3. 발생할 수 있음, 특히나 이번 처럼 음성데이터가 양성보다 압도적으로 많은 경우에는 불균형이 더욱 잦음, 이는 정확도는 어디까지나 진단 정답 비율만은 구하는 반면, 재현율은 양성 환자를 얼마나 찾아냇는지를 구하기 때문 
4. 실제 양성 환자를 음성으로 잘못 판정하는 오류를 최소화 하기 위해 재현율이 중요


## 7. k값에 따른 성능 비교 

In [11]:
def score(n):
    model = KNeighborsClassifier(n_neighbors=n)
    model.fit(x_train_filled,y_train)
    train = model.score(x_train_filled,y_train)
    test = model.score(x_test_filled,y_test)
    print(f"{n} /" ,round(train,2) ,"/", round(test,2))
    # print는 return을 빼도 됀다


In [12]:
k =[1,3,5,7,10,15,20,30]
for n in k:
    score(n)    


1 / 1.0 / 0.68
3 / 0.84 / 0.68
5 / 0.8 / 0.68
7 / 0.79 / 0.68
10 / 0.8 / 0.67
15 / 0.79 / 0.66
20 / 0.79 / 0.72
30 / 0.77 / 0.72


## k	학습 정확도	테스트 정확도
* 1	 	    1.0     /   0.68
* 3	        0.84    /   0.68	
* 5	        0.8     /   0.68	
* 7	        0.79    /   0.68	
* 10		0.8     /   0.67    
* 15	    0.79    /   0.66	
* 20	    0.79    /   0.72	
* 30	    0.77    /   0.72	

## 분석

1. 1인 경우가 학습 정확도가 가장 높음
2. 20, 30이 테스트 정확도가 가장 높음
3. k 값이 작으면 극소수의 가장 인접한 이웃 데이터만 참고하기에, 데이터의 전체적인 패턴이 아닌 개별 데이터의 이상치에 민감하게 반응함, 그로 인해  결정 경계가 지나치게 복잡해져 학습데이터에 대한 민감도는 올라가는 반면 일반화 성능은 떨어지는 과대적합이 발생
4. 반대로 k 값이 지나치게 커지면 너무 많은 데이터를 참고하여, 각 데이터의 세부적인 특성등을 반영하지 못함, 이로 인해 결정 경계가 지나치게 단순해져 학습 데이터와 일반화 성능 모두 하락하는 과소적합이 발생
5. 학습 정확도만을 본다면 1번의 케이스 처럼 학습데이터를 단순히 외워버리는 과대적합 상태를 걸러낼 수 없기에 일반화 성능의 신뢰성에 문제가 생김 고로 최적의 k값을 선택해야함

## 8. StandardScaler 적용

In [13]:

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_filled)
x_test_scaled = scaler.transform(x_test_filled)

x_train_scaled_df = pd.DataFrame(x_train_scaled,columns=x.columns)
#  데이터프레임 반환
x_train_scaled_df[["insu","pedi"]].describe()

,insu,pedi
count,6.140000e+02,6.140000e+02
mean,-4.339634e-18,-1.099374e-16
std,1.000815e+00,1.000815e+00
min,-1.559139e+00,-1.192094e+00
25%,-2.249692e-01,-7.042617e-01
50%,-1.614373e-01,-2.876344e-01
75%,-9.790537e-02,4.903223e-01
max,7.703814e+00,5.610293e+00


In [14]:
model_scaled = KNeighborsClassifier(n_neighbors=5)
model_scaled.fit(x_train_scaled,y_train)
pred_scaled = model_scaled.predict(x_test_scaled)
print("학습 정확도:", model_scaled.score(x_train_scaled,y_train))
print("테스트 정확도:", model_scaled.score(x_test_scaled,y_test))
print("재현율:", recall_score(y_test,pred_scaled))

학습 정확도: 0.8289902280130294
테스트 정확도: 0.7532467532467533
재현율: 0.6111111111111112


In [15]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(x_train_filled,y_train)
print("학습 정확도 :",model.score(x_train_filled,y_train))

pred = model.predict(x_test_filled)
print("테스트 정확도 :",model.score(x_test_filled,y_test))
print("재현율:", recall_score(y_test,pred))

학습 정확도 : 0.8013029315960912
테스트 정확도 : 0.6753246753246753
재현율: 0.5


In [16]:
print(x_train_scaled_df.mean())
print(x_train_scaled_df.std())

preg   -6.943414e-17
plas   -1.099374e-16
pres    3.095606e-16
skin   -3.471707e-17
insu   -4.339634e-18
mass    1.649061e-16
pedi   -1.099374e-16
age    -1.084908e-16
dtype: float64
preg    1.000815
plas    1.000815
pres    1.000815
skin    1.000815
insu    1.000815
mass    1.000815
pedi    1.000815
age     1.000815
dtype: float64


## 분석

1. 평균 0 표준편차 1
2. 학습 정확도: 0.801 → 0.829 (약 2.8%p 상승)
   테스트 정확도: 0.675 → 0.753 (약 7.8%p 상승)
3. KNN은 데이터 간의 거리를 계산.분류 하는 알고리즘, 스케일링 전에는 값의 범위가 큰 특성이 과도하게 반영되고 작은 특성은 영향이 과소 반영 혹은 무시되는 경향이 있었으나, 표준화를 통해 동일한 스케일을 적용함으로서, 모든 특성의 거리 계산이 균등하게 반영되어 정확도가 향상 



In [17]:
minmax = MinMaxScaler()
x_train_minmax = minmax.fit_transform(x_train_filled)
x_test_minmax = minmax.transform(x_test_filled)

x_train_minmax_df = pd.DataFrame(x_train_minmax, columns=x.columns)
x_train_minmax_df.describe()

,preg,plas,pres,skin,insu,mass,pedi,age
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,0.224660,0.459238,0.491225,0.239591,0.168320,0.291374,0.175246,0.206107
std,0.194950,0.209817,0.125256,0.096651,0.108045,0.139553,0.147127,0.197224
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.058824,0.300699,0.408163,0.195652,0.144033,0.192740,0.071715,0.050000
50%,0.176471,0.426573,0.489796,0.239130,0.150892,0.290389,0.132962,0.133333
75%,0.352941,0.587413,0.571429,0.271739,0.157750,0.374233,0.247327,0.333333
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [18]:
model_minmax = KNeighborsClassifier(n_neighbors=5)
model_minmax.fit(x_train_minmax,y_train)
model_minmax.score(x_train_minmax,y_train)
model_minmax.score(x_test_minmax,y_test)
pred_minmax = model_minmax.predict(x_test_minmax)
recall_score(y_test, pred)
print("학습 정확도 :", model_minmax.score(x_train_minmax,y_train))
print("테스트 정확도 :", model_minmax.score(x_test_minmax,y_test)) 
print("재현율:", recall_score(y_test, pred_minmax))     

학습 정확도 : 0.8322475570032574
테스트 정확도 : 0.7337662337662337
재현율: 0.5555555555555556


In [19]:
print("학습 정확도 :",round(model.score(x_train_filled,y_train),2))
print("테스트 정확도 :",round(model.score(x_test_filled,y_test),2))
print("재현율:", round(recall_score(y_test,pred),2))

print("표준화 학습 정확도:", round(model_scaled.score(x_train_scaled,y_train),2))
print("표준화 테스트 정확도:", round(model_scaled.score(x_test_scaled,y_test),2))
print("표준화 재현율:", round(recall_score(y_test,pred_scaled),2))

print("정규화 학습 정확도 :", round(model_minmax.score(x_train_minmax,y_train),2))
print("정규화 테스트 정확도 :", round(model_minmax.score(x_test_minmax,y_test),2)) 
print("정규화 재현율:", round(recall_score(y_test, pred_minmax),2))     


학습 정확도 : 0.8
테스트 정확도 : 0.68
재현율: 0.5
표준화 학습 정확도: 0.83
표준화 테스트 정확도: 0.75
표준화 재현율: 0.61
정규화 학습 정확도 : 0.83
정규화 테스트 정확도 : 0.73
정규화 재현율: 0.56


* KNN 전처리 방법	학습 정확도	테스트 정확도	재현율
* 스케일링 없음	    0.8         0.68          0.5		
* StandardScaler  0.83        0.75          0.61			
* MinMaxScaler    0.83        0.73          0.56   			

## 분석

1. 표준화의 테스트 정확도가 가장 높음
2. 표준화의 재현율이 가장 높음
3. 다를 수 있음 정확도는 올바른 진단을 내린 비율만을 의미하고, 재현율은 실제 양성 데이터를 찾아낸 비율이기에 서로 다르다. 특히나 이번 처럼 클래스 불균형(음성 65% 양성 35%)이 있는 경우에는 정확도와 재현율이 반비례 하는 경향이 있음. 여기서 전처리 방법의 차이가 더 해진다면, 각 방법에 따라 데이터간 거리 계산이 달라지는 추가 변수가 생김, 이로 인해 어떤 방식은 다수인 클래스(음성)을 더 잘맞추게 되어 정확도를 높게 잡는 반면, 다른 방식은 소수인 양성 데이터를 잘 잡아내 재현율이 높게 나올 수 있다. 이번 실험에서는 표준화가 둘다 가장 우수한 결과를 만들어냄
4. 스캐일링은 모든 특성의 크기를 균등하게 맞출 뿐, 데이터 자체의 한계나 알고리즘의 특성까지 해결하지는 못함, 예를 들어 예측에 불필요한 노이즈의 영향력도 커져 오히려 계산을 방해할 수도 있고, 이상치 있는 경우 데이터 분포가 왜곡되어 오히려 성능이 하락하기도 함. 마지막으로 특정 모델들은 스캐일링의 영향을 받지 않기에 이러한 결과가 나온다. 

## 10.종합 — KNN 설정 선택 및 최종 분석

* k값 실험

In [20]:
def score(n):
    model = KNeighborsClassifier(n_neighbors=n)
    model.fit(x_train_filled,y_train)
    train = model.score(x_train_filled,y_train)
    test = model.score(x_test_filled,y_test)
    pred = model.predict(x_test_filled)
    recall_score(y_test,pred)
    print(f"{n} 학습 정확도:" ,round(train,2) ,
          "테스트 정확도:", round(test,2),
          "재현율:", round(recall_score(y_test,pred),2))

    k =[1,3,5,7,10,15,20,30]
for n in k:
    score(n)    


1 학습 정확도: 1.0 테스트 정확도: 0.68 재현율: 0.54
3 학습 정확도: 0.84 테스트 정확도: 0.68 재현율: 0.52
5 학습 정확도: 0.8 테스트 정확도: 0.68 재현율: 0.5
7 학습 정확도: 0.79 테스트 정확도: 0.68 재현율: 0.48
10 학습 정확도: 0.8 테스트 정확도: 0.67 재현율: 0.44
15 학습 정확도: 0.79 테스트 정확도: 0.66 재현율: 0.44
20 학습 정확도: 0.79 테스트 정확도: 0.72 재현율: 0.5
30 학습 정확도: 0.77 테스트 정확도: 0.72 재현율: 0.48


In [21]:
def score(n):
    model = KNeighborsClassifier(n_neighbors=n)
    model.fit(x_train_scaled,y_train)
    train = model.score(x_train_scaled,y_train)
    test = model.score(x_test_scaled,y_test)
    pred_scaled = model.predict(x_test_scaled)
    recall_score(y_test,pred_scaled)
    print(f"{n} 표준화 학습 정확도:" ,round(train,2) ,
          "표준화 테스트 정확도:", round(test,2),
          "표준화 재현율:", round(recall_score(y_test,pred_scaled),2))

    k =[1,3,5,7,10,15,20,30]
for n in k:
    score(n)    

1 표준화 학습 정확도: 1.0 표준화 테스트 정확도: 0.72 표준화 재현율: 0.56
3 표준화 학습 정확도: 0.85 표준화 테스트 정확도: 0.72 표준화 재현율: 0.61
5 표준화 학습 정확도: 0.83 표준화 테스트 정확도: 0.75 표준화 재현율: 0.61
7 표준화 학습 정확도: 0.82 표준화 테스트 정확도: 0.73 표준화 재현율: 0.56
10 표준화 학습 정확도: 0.8 표준화 테스트 정확도: 0.74 표준화 재현율: 0.54
15 표준화 학습 정확도: 0.8 표준화 테스트 정확도: 0.74 표준화 재현율: 0.56
20 표준화 학습 정확도: 0.79 표준화 테스트 정확도: 0.76 표준화 재현율: 0.56
30 표준화 학습 정확도: 0.79 표준화 테스트 정확도: 0.74 표준화 재현율: 0.52


In [22]:
def score(n):
    model = KNeighborsClassifier(n_neighbors=n)
    model.fit(x_train_minmax,y_train)
    train = model.score(x_train_minmax,y_train)
    test = model.score(x_test_minmax,y_test)
    pred_minmax = model.predict(x_test_minmax)
    recall_score(y_test,pred_minmax)
    print(f"{n} 정규화 학습 정확도:" ,round(train,2) ,
          "정규화 테스트 정확도:", round(test,2),
          "정규화 재현율:", round(recall_score(y_test,pred_minmax),2))

    k =[1,3,5,7,10,15,20,30]
for n in k:
    score(n)    

1 정규화 학습 정확도: 1.0 정규화 테스트 정확도: 0.7 정규화 재현율: 0.56
3 정규화 학습 정확도: 0.87 정규화 테스트 정확도: 0.74 정규화 재현율: 0.59
5 정규화 학습 정확도: 0.83 정규화 테스트 정확도: 0.73 정규화 재현율: 0.56
7 정규화 학습 정확도: 0.83 정규화 테스트 정확도: 0.73 정규화 재현율: 0.56
10 정규화 학습 정확도: 0.79 정규화 테스트 정확도: 0.73 정규화 재현율: 0.46
15 정규화 학습 정확도: 0.8 정규화 테스트 정확도: 0.73 정규화 재현율: 0.54
20 정규화 학습 정확도: 0.79 정규화 테스트 정확도: 0.73 정규화 재현율: 0.46
30 정규화 학습 정확도: 0.79 정규화 테스트 정확도: 0.74 정규화 재현율: 0.5


## K값 실험

1. 일반

| k | 학습 정확도 | 테스트 정확도 | 재현율 |
| :---: | :---: | :---: | :---: |
| 1 | 1.0 | 0.68 | 0.54 |
| 3 | 0.84 | 0.68 | 0.52 |
| 5 | 0.8 | 0.68 | 0.5 |
| 7 | 0.79 | 0.68 | 0.48 |
| 10 | 0.8 | 0.67 | 0.44 |
| 15 | 0.79 | 0.66 | 0.44 |
| 20 | 0.79 | 0.72 | 0.5 |
| 30 | 0.77 | 0.72 | 0.48 |


2. 표준화

| k | 학습 정확도 | 테스트 정확도 | 재현율 |
| :---: | :---: | :---: | :---: |
| 1 | 1.0 | 0.72 | 0.56 |
| 3 | 0.85 | 0.72 | 0.61 |
| 5 | 0.83 | 0.75 | 0.61 |
| 7 | 0.82 | 0.73 | 0.56 |
| 10 | 0.8 | 0.74 | 0.54 |
| 15 | 0.8 | 0.74 | 0.56 |
| 20 | 0.79 | 0.76 | 0.56 |
| 30 | 0.79 | 0.74 | 0.52 |

3. 정규화

| k | 학습 정확도 | 테스트 정확도 | 재현율 |
| :---: | :---: | :---: | :---: |
| 1 | 1.0 | 0.7 | 0.56 |
| 3 | 0.87 | 0.74 | 0.59 |
| 5 | 0.83 | 0.73 | 0.56 |
| 7 | 0.83 | 0.73 | 0.56 |
| 10 | 0.79 | 0.73 | 0.46 |
| 15 | 0.8 | 0.73 | 0.54 |
| 20 | 0.79 | 0.73 | 0.46 |
| 30 | 0.79 | 0.74 | 0.5 |



## 최종 설정 선택

1. 표준화 방법
2. K값 5

In [23]:
model_scaled = KNeighborsClassifier(n_neighbors=5)
model_scaled.fit(x_train_scaled,y_train)
pred_scaled = model_scaled.predict(x_test_scaled)

print("혼동행렬:", confusion_matrix(y_test, pred_scaled))
print("정밀도:", precision_score(y_test, pred_scaled))
print("F1 점수:", round(f1_score(y_test, pred),2))


혼동행렬: [[83 17]
 [21 33]]
정밀도: 0.66
F1 점수: 0.52


## 선택한 설정으로 KNN 학습 및 평가

1. 표준화 학습 정확도: 0.83
2. 표준화 테스트 정확도: 0.75
3. 혼동행렬: [[83 17]
            [21 33]]
4. 정밀도: 0.66
5. 표준화 재현율: 0.61
6. F1 점수: 0.52

## 최종 분석 보고서

1. 표준화 방법과 k=5값을 선택
2. 가장 높은 테스트 정확도와 재현율을 보여줌 - 이번 실험에선 재현율이 가장 중요
3. 전반적으로 테스트 정확도와 재현율이 상승함
4. k값의 적정구간인 5~7값 사이에서 정확도와 재현율이 상승햇지만 값이 너무 낮아 과대 적합에 가까울수록 정확도와 재현율이 낮아졋으며, 반대로 적정치를 넘어 과소적합으로 갈 수록 마찬가지로 결과값의 수치가 낮아짐
5. 이번 실험의 경우에는 양성 환자를 정확하게 잡아내는 것이 가장 중요하기에 재현율을 가장 중요하게 고려함
6. 테스트 데이터에도 해당 기준을 적용하면 테스트 데이터를 미리 학습하는 데이터 누수가 발생하며, 또한 실전에선 데이터를 1개씩 입력받기에 테스트 데이터에 해당 기준을 적용하기가 어려움, 결론적으로 이러한 이유들로 인해 모델의 일반화 성능의 신뢰성이 하기에 해당 기준을 학습 데이터에만 적용해야 한다.
7. 이번 모델과 실험의 경우에는 데이터 편차값의 차이를 적절히 조절하고 과대적합이나 과소적합이 일어나지 않게 방지함으로서 참고를 할 데이터의 특성을 적절하게 고려하면서도 특정한 곳으로 편향되지 않게 균형을 잡는 것이 가장 중요.

